In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

import gpjax as gpx

In [ ]:
cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

---

In [ ]:
from dmpe.utils.density_estimation import build_grid, DensityEstimate
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

## GP modeling:

In [ ]:
def perform_GPR(x, y, xtest):

    # construct prior
    kernel = gpx.kernels.RBF()
    meanf = gpx.mean_functions.Zero()
    prior = gpx.gps.Prior(mean_function=meanf, kernel=kernel)

    # construct posterior based on the given dataset
    D = gpx.Dataset(X=x, y=y)
    likelihood = gpx.likelihoods.Gaussian(num_datapoints=D.n)
    posterior = prior * likelihood

    # calibrate model
    posterior, history = gpx.fit_scipy(
        model=posterior,
        objective=lambda p, d: -gpx.objectives.conjugate_mll(p, d),
        train_data=D,
    )
    latent_dist = posterior.predict(xtest, train_data=D, return_covariance_type="diagonal")
    predictive_dist = posterior.likelihood(latent_dist)

    predictive_mean = predictive_dist.mean
    predictive_std = jnp.sqrt(predictive_dist.variance)

    return predictive_mean, predictive_std

In [ ]:
key = jax.random.key(1)
n = 20
noise = 0.05

key, subkey = jax.random.split(key)
x = jax.random.uniform(key=key, minval=-3.0, maxval=3.0, shape=(n,)).reshape(-1, 1)
f = lambda x: jnp.sin(4 * x) + jnp.cos(2 * x)
signal = f(x)
y = signal + jax.random.normal(subkey, shape=signal.shape) * noise

D = gpx.Dataset(X=x, y=y)

xtest = jnp.linspace(-3.5, 3.5, 500).reshape(-1, 1)
ytest = f(xtest)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 2.5))
ax.plot(x, y, "o", label="Observations", color=cols[0])
ax.plot(xtest, ytest, label="Latent function", color=cols[1])
ax.legend(loc="best")

In [ ]:
mean_gp, sigma_gp = perform_GPR(x, y, xtest)
fig, ax = plt.subplots(figsize=(7.5, 2.5))
ax.plot(x, y, "x", label="Observations", color=cols[0], alpha=0.5)
ax.fill_between(
    xtest.squeeze(),
    mean_gp - 2 * sigma_gp,
    mean_gp + 2 * sigma_gp,
    alpha=0.2,
    label="Two sigma",
    color=cols[1],
)
ax.plot(
    xtest,
    mean_gp - 2 * sigma_gp,
    linestyle="--",
    linewidth=1,
    color=cols[1],
)
ax.plot(
    xtest,
    mean_gp + 2 * sigma_gp,
    linestyle="--",
    linewidth=1,
    color=cols[1],
)
ax.plot(
    xtest, ytest, label="Latent function", color=cols[0], linestyle="--", linewidth=2
)
ax.plot(xtest, mean_gp, label="Predictive mean", color=cols[1])
ax.legend(loc="center left", bbox_to_anchor=(0.975, 0.5))

## GP excitation:

- consider combining `gp_model` and `train_data`

In [ ]:
from dmpe.algorithms.algorithm_utils import default_dmpe_parameterization
from dmpe.models.models import NeuralEulerODEPendulum

from dmpe.excitation.excitation_utils import soft_penalty

from dmpe.evaluation.plotting_utils import plot_feature_combinations, plot_sequence

In [ ]:
from admpe.gp_excitation.algorithm import excite_with_gp_uncertainty

In [ ]:
env, penalty_function, featurize, env_params = setup_pendulum_env()

penalty_function = lambda x, u: soft_penalty(a=x, a_max=1, penalty_order=2) * 1e3 + soft_penalty(
    a=u, a_max=1, penalty_order=2
) * 1e3

In [ ]:
env

In [ ]:
exp_params, proposed_actions, loader_key, expl_key, process_noise_key = default_dmpe_parameterization(env, seed=0, n_time_steps=2000, featurize=featurize, model_class=NeuralEulerODEPendulum)

In [ ]:
exp_params["alg_params"]["loss_type"] = "trajectory"

In [ ]:
observations, actions, model, gp_model, losses, proposed_actions, callback_out = excite_with_gp_uncertainty(env, exp_params, proposed_actions, loader_key, expl_key, process_noise_key)

In [ ]:
plot_sequence(
    observations[:-1],
    actions,
    env.tau,
    env.obs_description,
    env.action_description,
)

In [ ]:
plot_sequence(
    observations[1800:2000],
    actions[1800:2000],
    env.tau,
    env.obs_description,
    env.action_description,
)

In [ ]:
plot_feature_combinations(jnp.hstack([
    observations[:-1],
    actions,
]), list(env.obs_description) + list(env.action_description));
plt.show()

- visualize current state of the GP?
- why is it oversampling so heavily?
- build a density estimation object of the posterior variance

In [ ]:
x_plot = build_grid(dim=3, low=-1.05, high=1.05, points_per_dim=20)
x_plot.shape

In [ ]:
var = gp_model.predict_posterior_variance(x_plot)

In [ ]:
#var = var.reshape((20, 20, 20, 2))

In [ ]:
DensityEstimate(p=var[:, 0], z_g=x_plot, bandwidth=0.1, n_observations=0).visualize()

In [ ]:
assert jnp.all(var[:, 0] == var[:, 1])

In [ ]:
gp_model.posteriors